# Fundamentos de Ingeniería de Datos
## TRANSFORMACIÓN: de EXTRACTED a TRANSFORMED

```
INGESTA            EXTRACCIÓN                  TRANSFORMACIÓN (este notebook)
raw_autos     →    extracted_autos        →    diagnosticar · limpiar · estandarizar · tipar · enriquecer
(SQLite)           (SQLite)                    → Processed/transformed_autos.parquet + bitácora + lineage
```

### ¿Qué es la capa TRANSFORMED?

Son los datos de `extracted_autos` **limpios, con tipos correctos y listos para analizar**. Es la primera capa en la que **sí cambiamos** los datos.

| Capa | Qué contiene | En la industria (*medallion*) |
|---|---|---|
| RAW | Datos tal como llegaron | Bronze |
| EXTRACTED | Los mismos datos + metadatos de extracción | Bronze |
| **TRANSFORMED** | **Datos limpios, tipados y enriquecidos** | **Silver** |
| SERVING | Esquema estrella para análisis | Gold |

**Las transformaciones se deciden por el uso de los datos:**

- **Modelo** ("¿Cuánto vale mi carro?"): predice `price` a partir de las características del auto. No acepta nulos y necesita categorías consistentes.
- **Tablero** ("¿Cuánto vale el carro que quiero comprar?"): filtros y gráficas por marca, modelo, año, millaje, etc.

> **Regla de oro:** cada capa lee **solo** de la anterior. Este notebook lee `extracted_autos` (nunca `raw_autos`) en **modo solo lectura**.

### Configuración del entorno

In [1]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import (
    PROYECTO_BASE, DB_PATH, PROCESSED_PATH, TRANSFORMACION_LOGS, TRANSFORMACION_REPORTES,
    inspeccionar_dataset, listar_tablas, tamano_mb, ruta_relativa
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {os.path.basename(PROYECTO_BASE)}/')
print()


utils.py importado correctamente
Proyecto base: ING_BD_ProjectF/



In [2]:
# ── Librerías, nombres y archivos de esta capa ────────────────────────────

import re
import json
import sqlite3
import numpy as np
import pandas as pd
from datetime import datetime

pd.set_option('display.max_columns', None)


ETIQUETA          = 'autos'
TABLA_EXTRACTED   = f'extracted_{ETIQUETA}'                 # origen (SQLite)
NOMBRE_DESTINO    = f'transformed_{ETIQUETA}'               # destino (Parquet)
VERSION_TRANSFORM = 'v1'                                    # súbela si cambias las reglas de limpieza

PARQUET_DESTINO = f'{PROCESSED_PATH}/{NOMBRE_DESTINO}.parquet'
LOG_PATH        = f'{TRANSFORMACION_LOGS}/transformacion_log.jsonl'
LINEAGE_PATH    = f'{TRANSFORMACION_LOGS}/lineage_{NOMBRE_DESTINO}.json'
BITACORA_PATH   = f'{TRANSFORMACION_REPORTES}/bitacora_{NOMBRE_DESTINO}.csv'
CALIDAD_PATH    = f'{TRANSFORMACION_REPORTES}/calidad_{NOMBRE_DESTINO}.csv'

for carpeta in (PROCESSED_PATH, TRANSFORMACION_LOGS, TRANSFORMACION_REPORTES):
    os.makedirs(carpeta, exist_ok=True)

print()
print(f'Origen   : {TABLA_EXTRACTED}  (SQLite, solo lectura)')
print(f'Destino  : {ruta_relativa(PARQUET_DESTINO)}')
print()


Origen   : extracted_autos  (SQLite, solo lectura)
Destino  : Transformacion_de_Datos/Processed/transformed_autos.parquet



---
## PASO 1 — Leer el origen: `extracted_autos`

Leemos en **solo lectura** y separamos las columnas en dos grupos:

- **Columnas de negocio:** las del dataset (las vamos a limpiar).
- **Metadatos** (empiezan con `_`): **no se tocan**. `_row_hash` permite rastrear cada fila limpia hasta su origen (*lineage*).

In [3]:
# ── Leer extracted_autos en modo solo lectura ──────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()

COLS_META    = [c for c in df_ext.columns if c.startswith('_')]
COLS_NEGOCIO = [c for c in df_ext.columns if not c.startswith('_')]

print()
print(f'Columnas de negocio : {COLS_NEGOCIO}')
print(f'Metadatos           : {COLS_META}')

inspeccionar_dataset(df_ext, nombre=TABLA_EXTRACTED)


Columnas de negocio : ['manufacturer', 'model', 'year', 'mileage', 'engine', 'transmission', 'drivetrain', 'fuel_type', 'mpg', 'exterior_color', 'interior_color', 'accidents_or_damage', 'one_owner', 'personal_use_only', 'seller_name', 'seller_rating', 'driver_rating', 'driver_reviews_num', 'price_drop', 'price']
Metadatos           : ['_batch_id', '_extracted_at', '_source_table', '_row_hash']

  DATASET: extracted_autos
  Filas    : 200,000
  Columnas : 24

  Tipos de datos y valores nulos:
    manufacturer                 str          nulos: 0 (0.0%)
    model                        str          nulos: 0 (0.0%)
    year                         int64        nulos: 0 (0.0%)
    mileage                      float64      nulos: 126 (0.1%)
    engine                       str          nulos: 3887 (1.9%)
    transmission                 str          nulos: 2545 (1.3%)
    drivetrain                   str          nulos: 5593 (2.8%)
    fuel_type                    str          nulos: 5977

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price,_batch_id,_extracted_at,_source_table,_row_hash
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0,1,2026-11-24T11:17:50,raw_autos,a5f41d2a94483372
1,Acura,ILX Hybrid 1.5L,2013,61539.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,NaN,1.0,1.0,1.0,Napleton's Palatine Subaru,4.6,4.4,12.0,NaN,16950.0,1,2026-11-24T11:17:50,raw_autos,d490101e1b4b1443
2,Acura,NSX Base,2017,15904.0,Twin Turbo Gas/Electric V-6 3.5 L/213,Automatic,All-wheel Drive,Hybrid,21-22,130R White,Ebony,0.0,0.0,1.0,Piazza Acura / Honda of Reading,4.7,4.8,17.0,2111.0,139876.0,1,2026-11-24T11:17:50,raw_autos,7497d846a2e6c6e0


---
## PASO 2 — Diagnóstico de calidad (antes)

**No se limpia lo que no se ha medido.** Antes de transformar, medimos los problemas; al final repetimos las mismas mediciones y comparamos.

| Dimensión | Pregunta | Cómo la medimos |
|---|---|---|
| **Completitud** | ¿Faltan datos? | Nulos por columna |
| **Unicidad** | ¿Hay anuncios repetidos? | Duplicados exactos |
| **Validez** | ¿Los valores tienen sentido? | Rangos de precio, año y millaje |
| **Estandarización** | ¿Un mismo valor se escribe de una sola forma? | Variantes distintas en transmisión, tracción, combustible y colores |
| **Consistencia** | ¿Los datos cuadran entre sí? | Autos eléctricos con cilindros en el motor |

### Roles de las columnas

| Rol | Columnas | Trato |
|---|---|---|
| **Variable objetivo** | `price` | Lo que predice el modelo y analiza el tablero |
| **Características del auto** | marca, modelo, año, millaje, motor, transmisión, tracción, combustible, rendimiento, colores, accidentes, un solo dueño, uso personal | Entradas del modelo y filtros del tablero: **no pueden quedar nulas** |
| **Precio relacionado** | `price_drop` | Solo para el tablero: se conoce *después* de publicar el anuncio, el modelo no debe usarla |
| **Informativas** | `seller_name`, `seller_rating`, `driver_rating`, `driver_reviews_num` | **Solo para el tablero.** Describen al vendedor y las reseñas, no al auto: el usuario no las puede capturar en el formulario. **No se imputan** (se dejan nulas cuando no hay dato) |

In [4]:
# ── Reglas del dataset ─────────────────────────────────────────────────────

PRECIO_MIN, PRECIO_MAX   = 1_000, 200_000   # USD: abajo son precios simbólicos; arriba, superdeportivos o errores
ANIO_MIN,   ANIO_MAX     = 2000, 2024       # antes de 2000 son clásicos: se valúan con otra lógica
MILLAJE_MIN, MILLAJE_MAX = 0, 500_000       # 999,999 es un valor de relleno
ANIO_REFERENCIA          = 2023             # año en que se recopilaron los anuncios (para calcular antigüedad)

COLS_BOOLEANAS    = ['accidents_or_damage', 'one_owner', 'personal_use_only']
COLS_CATEGORICAS  = ['transmission', 'drivetrain', 'fuel_type', 'exterior_color', 'interior_color']
COLS_INFORMATIVAS = ['seller_name', 'seller_rating', 'driver_rating', 'driver_reviews_num']

# Columnas originales que describen al auto (todas menos las informativas)
COLS_AUTO_ORIGEN = [c for c in COLS_NEGOCIO if c not in COLS_INFORMATIVAS]

print()
print(f'Precio válido  : ${PRECIO_MIN:,} – ${PRECIO_MAX:,}')
print(f'Año válido     : {ANIO_MIN} – {ANIO_MAX}')
print(f'Millaje válido : {MILLAJE_MIN:,} – {MILLAJE_MAX:,}')
print()


Precio válido  : $1,000 – $200,000
Año válido     : 2000 – 2024
Millaje válido : 0 – 500,000



In [5]:
# ── Completitud: nulos por columna ─────────────────────────────────────────

nulos = df_ext[COLS_NEGOCIO].isna().sum()

pd.DataFrame({'nulos': nulos, 'pct': (nulos / len(df_ext) * 100).round(1)}).query('nulos > 0')

,nulos,pct
mileage,126,0.1
engine,3887,1.9
transmission,2545,1.3
drivetrain,5593,2.8
fuel_type,5977,3.0
mpg,37071,18.5
exterior_color,2355,1.2
interior_color,15031,7.5
accidents_or_damage,6332,3.2
one_owner,8265,4.1


In [6]:
# ── Unicidad y validez ─────────────────────────────────────────────────────

dup_exactos      = df_ext.duplicated(subset=COLS_NEGOCIO).sum()
precio_invalido  = ~df_ext['price'].between(PRECIO_MIN, PRECIO_MAX)
anio_invalido    = ~df_ext['year'].between(ANIO_MIN, ANIO_MAX)
millaje_invalido = ~df_ext['mileage'].between(MILLAJE_MIN, MILLAJE_MAX)       # los nulos también cuentan

print()
print(f'Duplicados exactos                          : {dup_exactos:,}')
print(f'Precio fuera de [${PRECIO_MIN:,}, ${PRECIO_MAX:,}]        : {precio_invalido.sum():,}')
print(f'Año fuera de [{ANIO_MIN}, {ANIO_MAX}]                    : {anio_invalido.sum():,}')
print(f'Millaje nulo o fuera de [{MILLAJE_MIN}, {MILLAJE_MAX:,}]     : {millaje_invalido.sum():,}')
print()

# Ejemplos de precios fuera de rango
df_ext.loc[precio_invalido, ['manufacturer', 'model', 'year', 'mileage', 'price']].sort_values('price').iloc[[0, 1, 2, -3, -2, -1]]


Duplicados exactos                          : 612
Precio fuera de [$1,000, $200,000]        : 377
Año fuera de [2000, 2024]                    : 1,536
Millaje nulo o fuera de [0, 500,000]     : 135



,manufacturer,model,year,mileage,price
3632,Audi,A4 2.0T Premium,2014,98881.0,1.0
17339,Buick,Enclave Premium,2017,115869.0,299.0
193283,Volkswagen,Jetta S,2019,120092.0,319.0
12867,BMW,750 iL,1996,121043.0,1750000.0
160897,Porsche,Carrera GT,2004,2361.0,1899999.0
159831,Porsche,918 Spyder Base (PDK),2015,2622.0,2099995.0


In [7]:
# ── Estandarización: ¿de cuántas formas se escribe cada categoría? ─────────

variantes_antes = df_ext[COLS_CATEGORICAS].nunique()

print()
for col in COLS_CATEGORICAS:
    print(f'{col:<16} {variantes_antes[col]:>6,} variantes   ej.: {df_ext[col].value_counts().index[:4].tolist()}')
print()


transmission        696 variantes   ej.: ['6-Speed Automatic', '8-Speed Automatic', 'Automatic CVT', 'Automatic']
drivetrain           28 variantes   ej.: ['Front-wheel Drive', 'All-wheel Drive', 'Four-wheel Drive', 'Rear-wheel Drive']
fuel_type            23 variantes   ej.: ['Gasoline', 'Hybrid', 'Diesel', 'E85 Flex Fuel']
exterior_color    4,732 variantes   ej.: ['Black', 'White', 'Gray', 'Silver']
interior_color    2,705 variantes   ej.: ['Black', 'Gray', 'Jet Black', 'Ebony']



In [8]:
# ── Consistencia: autos eléctricos con cilindros en el motor ───────────────
# Un eléctrico no tiene motor de combustión: si el texto del motor menciona cilindros, algo no cuadra.

cilindros_texto  = df_ext['engine'].str.extract(r'\b(?:V|I|H|W|Flat)-?\s?(\d{1,2})\b', flags=re.I, expand=False)
electrico_inconsistente = (df_ext['fuel_type'] == 'Electric') & cilindros_texto.notna()

print()
print(f'Eléctricos con cilindros en el motor : {electrico_inconsistente.sum():,}')
print()


Eléctricos con cilindros en el motor : 37



In [9]:
# ── Resumen del diagnóstico (línea base para comparar después) ─────────────

calidad_antes = pd.Series({
    'filas'                          : len(df_ext),
    'celdas_nulas_columnas_del_auto' : df_ext[COLS_AUTO_ORIGEN].isna().sum().sum(),
    'celdas_nulas_informativas'      : df_ext[COLS_INFORMATIVAS].isna().sum().sum(),
    'duplicados_exactos'             : dup_exactos,
    'precio_fuera_de_rango'          : precio_invalido.sum(),
    'anio_fuera_de_rango'            : anio_invalido.sum(),
    'millaje_nulo_o_fuera_de_rango'  : millaje_invalido.sum(),
    **{f'variantes_{c}': variantes_antes[c] for c in COLS_CATEGORICAS},
    'electricos_con_cilindros'       : electrico_inconsistente.sum(),
}, name='antes')

calidad_antes.to_frame()

,antes
filas,200000
celdas_nulas_columnas_del_auto,186091
celdas_nulas_informativas,66819
duplicados_exactos,612
precio_fuera_de_rango,377
anio_fuera_de_rango,1536
millaje_nulo_o_fuera_de_rango,135
variantes_transmission,696
variantes_drivetrain,28
variantes_fuel_type,23


---
## PASO 3 — Transformaciones

Aplicamos las reglas **una por una** sobre una copia (`df`) y anotamos cada una en la **bitácora**: qué se hizo, **por qué** (pensando en el modelo y el tablero) y cuántas filas o celdas cambió.

| # | Transformación | Tipo |
|---|---|---|
| 3.1 | Limpiar espacios en el texto | Normalización |
| 3.2 | Quitar duplicados exactos | Limpieza (unicidad) |
| 3.3 | Eliminar filas sin datos clave | Limpieza (completitud) |
| 3.4 | Rangos de precio, año y millaje | Limpieza (validez) |
| 3.5 | Estandarizar transmisión, tracción y combustible | Normalización |
| 3.6 | Color base | Normalización |
| 3.7 | Modelo base | Normalización |
| 3.8 | Imputar categorías desconocidas | Limpieza (completitud) |
| 3.9 | Motor: cilindrada, cilindros y turbo | Extracción de atributos + consistencia |
| 3.10 | Rendimiento: ciudad y carretera | Extracción de atributos |
| 3.11 | Rebaja y columnas informativas | Limpieza (completitud) |
| 3.12 | Tipos de dato | Tipado |
| 3.13 | Catálogo de marcas (*join*) | Enriquecimiento |
| 3.14 | Columnas derivadas | Enriquecimiento |
| 3.15 | Metadatos y orden final | Lineage |

In [10]:
# ── Bitácora de transformaciones ───────────────────────────────────────────

bitacora = []

def registrar(paso, descripcion, justificacion, filas_antes, filas_despues, afectadas):
    '''Anota un paso en la bitácora y lo muestra.'''
    bitacora.append({
        'paso'           : paso,
        'descripcion'    : descripcion,
        'justificacion'  : justificacion,
        'filas_antes'    : filas_antes,
        'filas_despues'  : filas_despues,
        'filas_afectadas': int(afectadas),
    })
    print(f'[{paso}] {descripcion}')
    print(f'       filas: {filas_antes:,} → {filas_despues:,}   (afectadas: {int(afectadas):,})')


df = df_ext.copy()        # trabajamos sobre una copia: df_ext queda intacto para comparar

print()
print(f'Punto de partida: {len(df):,} filas')
print()


Punto de partida: 200,000 filas



### 3.1 Limpiar espacios en el texto

Espacios al inicio o al final (`'Ford '`) o dobles espacios hacen que dos valores iguales se vean distintos al agrupar en el tablero o al codificar categorías en el modelo.

In [11]:
# ── 3.1 Limpiar columnas de texto ──────────────────────────────────────────

COLS_TEXTO = [c for c in COLS_NEGOCIO if df[c].dtype == 'object' or pd.api.types.is_string_dtype(df[c])]

def limpiar_texto(columna):
    '''Quita espacios al inicio y al final, y deja un solo espacio entre palabras.'''
    return columna.str.strip().str.replace(r'\s+', ' ', regex=True)

texto_antes    = df[COLS_TEXTO].copy()
df[COLS_TEXTO] = df[COLS_TEXTO].apply(limpiar_texto)
cambiadas      = (df[COLS_TEXTO].fillna('') != texto_antes.fillna('')).sum().sum()

print()
registrar('3.1', f'Quitar espacios extra en {len(COLS_TEXTO)} columnas de texto',
          'Valores iguales deben agruparse igual en el tablero y codificarse igual en el modelo',
          len(df), len(df), cambiadas)
print()


[3.1] Quitar espacios extra en 10 columnas de texto
       filas: 200,000 → 200,000   (afectadas: 149)



### 3.2 Quitar duplicados exactos (unicidad)

Si todas las columnas del dataset son iguales, es el **mismo anuncio repetido**: se conserva el primero. Se comparan solo `COLS_NEGOCIO`, porque los metadatos son distintos entre filas.

In [12]:
# ── 3.2 Duplicados exactos ─────────────────────────────────────────────────

antes = len(df)
df = df.drop_duplicates(subset=COLS_NEGOCIO, keep='first')

registrar('3.2', 'Eliminar duplicados exactos',
          'Un anuncio repetido pesaría doble en el entrenamiento del modelo y en los conteos del tablero',
          antes, len(df), antes - len(df))

[3.2] Eliminar duplicados exactos
       filas: 200,000 → 199,388   (afectadas: 612)


### 3.3 Eliminar filas sin datos clave (completitud)

| Columna | Decisión | Por qué |
|---|---|---|
| `mileage` | **Eliminar la fila** | Es una de las variables que más explica el precio; inventarla sesgaría la predicción |
| `accidents_or_damage`, `one_owner`, `personal_use_only` | **Eliminar la fila** | El usuario sí los captura en el formulario; imputarlos metería ruido justo en variables clave |

Las demás columnas con nulos se tratan más adelante (3.8 a 3.11), porque **sí** se pueden deducir de otros datos del auto o porque son informativas.

In [13]:
# ── 3.3 Nulos en columnas clave ────────────────────────────────────────────

antes = len(df)
df = df.dropna(subset=['mileage'])
registrar('3.3a', 'Eliminar filas sin millaje',
          'El millaje es clave para el precio: no se puede inventar sin sesgar el modelo',
          antes, len(df), antes - len(df))

antes = len(df)
df = df.dropna(subset=COLS_BOOLEANAS)
registrar('3.3b', f'Eliminar filas con nulos en {COLS_BOOLEANAS}',
          'Son entradas del formulario: imputarlas metería ruido en variables que el usuario sí conoce',
          antes, len(df), antes - len(df))

[3.3a] Eliminar filas sin millaje
       filas: 199,388 → 199,263   (afectadas: 125)


[3.3b] Eliminar filas con nulos en ['accidents_or_damage', 'one_owner', 'personal_use_only']
       filas: 199,263 → 191,073   (afectadas: 8,190)


### 3.4 Rangos de precio, año y millaje (validez)

| Regla | Por qué |
|---|---|
| Precio entre **$1,000 y $200,000** | Abajo hay precios simbólicos ($1, $299). Arriba hay superdeportivos reales y errores ($1,111,111). Ninguno es un auto que el usuario valúe, y distorsionarían el modelo |
| Año entre **2000 y 2024** | Los clásicos se **aprecian** con la edad: siguen la lógica contraria al resto y confundirían al modelo |
| Millaje entre **0 y 500,000** | 999,999 es un valor de relleno. 0 sí es válido: son autos seminuevos o de exhibición |

In [14]:
# ── 3.4 Rangos ─────────────────────────────────────────────────────────────

antes = len(df)
df = df[df['price'].between(PRECIO_MIN, PRECIO_MAX)]
registrar('3.4a', f'Eliminar precio fuera de [${PRECIO_MIN:,}, ${PRECIO_MAX:,}]',
          'Precios simbólicos, errores y superdeportivos distorsionan el modelo y no son autos que valúe el usuario',
          antes, len(df), antes - len(df))

antes = len(df)
df = df[df['year'].between(ANIO_MIN, ANIO_MAX)]
registrar('3.4b', f'Eliminar año fuera de [{ANIO_MIN}, {ANIO_MAX}]',
          'Los clásicos se aprecian con la edad: siguen otra lógica de precio que confundiría al modelo',
          antes, len(df), antes - len(df))

antes = len(df)
df = df[df['mileage'].between(MILLAJE_MIN, MILLAJE_MAX)]
registrar('3.4c', f'Eliminar millaje fuera de [{MILLAJE_MIN}, {MILLAJE_MAX:,}]',
          'Valores de relleno (999,999) que no son un millaje real',
          antes, len(df), antes - len(df))

[3.4a] Eliminar precio fuera de [$1,000, $200,000]
       filas: 191,073 → 190,747   (afectadas: 326)


[3.4b] Eliminar año fuera de [2000, 2024]
       filas: 190,747 → 189,854   (afectadas: 893)


[3.4c] Eliminar millaje fuera de [0, 500,000]
       filas: 189,854 → 189,845   (afectadas: 9)


### 3.5 Estandarizar transmisión, tracción y combustible

Cada categoría se escribe de decenas o cientos de formas (`'6-Speed Automatic'`, `'Automatic CVT'`, `'A'`…). Para el modelo cada variante sería una categoría distinta, y en el tablero un filtro inservible. Se reducen a pocas categorías con reglas de texto. Lo que no se reconoce queda como nulo y se imputa en 3.8.

| Columna | Categorías finales |
|---|---|
| `transmission` | Automatic · Manual · CVT |
| `drivetrain` | FWD · RWD · AWD · 4WD |
| `fuel_type` | Gasoline · Hybrid · Plug-In Hybrid · Diesel · Flex Fuel · Electric · Other |

In [15]:
# ── 3.5 Reglas de estandarización ──────────────────────────────────────────

def estandarizar_transmision(x):
    if pd.isna(x): return np.nan
    t = x.lower()
    if 'cvt' in t or 'ivt' in t or 'variable' in t:               return 'CVT'
    if 'manual' in t or 'm/t' in t or t == 'm':                   return 'Manual'
    if any(p in t for p in ['auto', 'a/t', 'speed', 'dual', 'dct', 'pdk', 'shiftronic', 'select']) or t == 'a':
        return 'Automatic'
    return np.nan                                                 # 'Not Specified', 'AWD', 'Other'…


def estandarizar_traccion(x):
    if pd.isna(x): return np.nan
    t = x.lower()
    if t.startswith('engine'):                                    return np.nan   # el texto del motor quedó aquí por error
    if 'front' in t or t == 'fwd':                                return 'FWD'
    if 'rear' in t or t == 'rwd':                                 return 'RWD'
    if 'all' in t or t == 'awd':                                  return 'AWD'
    if 'four' in t or t in ('4wd', '4x4'):                        return '4WD'
    return np.nan                                                 # 'Unknown', '2WD', '4x2'


def estandarizar_combustible(x):
    if pd.isna(x): return np.nan
    t = x.lower()
    if 'plug-in' in t:                                            return 'Plug-In Hybrid'
    if 'hybrid' in t:                                             return 'Hybrid'
    if 'diesel' in t:                                             return 'Diesel'
    if 'e85' in t or 'flex' in t:                                 return 'Flex Fuel'
    if 'electric' in t:                                           return 'Electric'
    if 'gasoline' in t or 'unleaded' in t or t == 'g':            return 'Gasoline'
    if 'natural gas' in t or 'gaseous' in t:                      return 'Other'
    return np.nan                                                 # 'B', 'Unspecified', 'Other'


reglas = {'transmission': estandarizar_transmision,
          'drivetrain'  : estandarizar_traccion,
          'fuel_type'   : estandarizar_combustible}

for col, regla in reglas.items():
    original = df[col]
    df[col]  = original.map(regla)
    sin_reconocer = (original.notna() & df[col].isna()).sum()
    registrar('3.5', f'Estandarizar {col}: {original.nunique():,} variantes → {df[col].nunique()} categorías '
                     f'({sin_reconocer:,} valores no reconocibles → nulo)',
              'Pocas categorías consistentes: el modelo las codifica bien y el tablero las usa como filtro',
              len(df), len(df), (original.fillna('') != df[col].fillna('')).sum())

pd.DataFrame({c: df[c].value_counts(dropna=False) for c in reglas}).fillna(0).astype(int)

[3.5] Estandarizar transmission: 539 variantes → 3 categorías (100 valores no reconocibles → nulo)
       filas: 189,845 → 189,845   (afectadas: 162,557)


[3.5] Estandarizar drivetrain: 26 variantes → 4 categorías (16 valores no reconocibles → nulo)
       filas: 189,845 → 189,845   (afectadas: 182,555)


[3.5] Estandarizar fuel_type: 20 variantes → 7 categorías (352 valores no reconocibles → nulo)
       filas: 189,845 → 189,845   (afectadas: 5,167)


,transmission,drivetrain,fuel_type
4WD,0,40096,0
AWD,0,58616,0
Automatic,151775,0,0
CVT,30186,0,0
Diesel,0,0,7086
Electric,0,0,3961
FWD,0,62394,0
Flex Fuel,0,0,4757
Gasoline,0,0,161142
Hybrid,0,0,7430


### 3.6 Color base

Los colores son nombres comerciales: `'Summit White'`, `'Oxford White'`, `'Bright White Clearcoat'` son el mismo blanco. Se busca primero un **color directo** en el texto (`white`, `black`, `blue`…) y, si no hay, un **sinónimo** (`ebony` → Black, `charcoal` → Gray…). Lo que no se reconoce queda como `'Other'` y los nulos como `'Unknown'`: el color no se puede deducir de otros datos.

In [16]:
# ── 3.6 Color base ─────────────────────────────────────────────────────────

COLORES_DIRECTOS = {
    'white': 'White', 'black': 'Black', 'silver': 'Silver', 'gray': 'Gray', 'grey': 'Gray',
    'blue': 'Blue', 'red': 'Red', 'green': 'Green', 'brown': 'Brown', 'beige': 'Beige',
    'tan': 'Beige', 'gold': 'Gold', 'yellow': 'Yellow', 'orange': 'Orange', 'purple': 'Purple',
}
SINONIMOS = {
    'ebony': 'Black', 'onyx': 'Black', 'jet': 'Black', 'obsidian': 'Black', 'carbon': 'Black',
    'charcoal': 'Gray', 'graphite': 'Gray', 'granite': 'Gray', 'gun': 'Gray', 'gunmetal': 'Gray',
    'magnetic': 'Gray', 'steel': 'Gray', 'titanium': 'Gray', 'slate': 'Gray', 'ash': 'Gray',
    'platinum': 'Silver', 'ice': 'Silver', 'sterling': 'Silver', 'quicksilver': 'Silver',
    'alpine': 'White', 'polar': 'White', 'glacier': 'White', 'oxford': 'White', 'ivory': 'White',
    'pearl': 'White', 'snow': 'White', 'chalk': 'White',
    'sapphire': 'Blue', 'navy': 'Blue', 'indigo': 'Blue', 'cobalt': 'Blue', 'aqua': 'Blue',
    'burgundy': 'Red', 'maroon': 'Red', 'ruby': 'Red', 'crimson': 'Red', 'cherry': 'Red', 'wine': 'Red',
    'bronze': 'Brown', 'copper': 'Brown', 'mocha': 'Brown', 'espresso': 'Brown', 'chestnut': 'Brown',
    'saddle': 'Brown', 'cognac': 'Brown', 'walnut': 'Brown',
    'champagne': 'Beige', 'sand': 'Beige', 'khaki': 'Beige', 'parchment': 'Beige', 'cream': 'Beige',
    'stone': 'Beige', 'cashmere': 'Beige', 'oyster': 'Beige', 'almond': 'Beige', 'linen': 'Beige',
}

def color_base(x):
    if pd.isna(x): return 'Unknown'
    palabras = re.findall(r'[a-z]+', x.lower())
    for p in palabras:
        if p in COLORES_DIRECTOS: return COLORES_DIRECTOS[p]
    for p in palabras:
        if p in SINONIMOS:        return SINONIMOS[p]
    return 'Other'


for col in ['exterior_color', 'interior_color']:
    original = df[col]
    df[col]  = original.map(color_base)
    registrar('3.6', f'Color base en {col}: {original.nunique():,} variantes → {df[col].nunique()} colores',
              'Miles de nombres comerciales no sirven como filtro ni como categoría del modelo',
              len(df), len(df), original.notna().sum())

pd.DataFrame({c: df[c].value_counts() for c in ['exterior_color', 'interior_color']}).fillna(0).astype(int)

[3.6] Color base en exterior_color: 4,455 variantes → 15 colores
       filas: 189,845 → 189,845   (afectadas: 187,662)


[3.6] Color base en interior_color: 2,562 variantes → 14 colores
       filas: 189,845 → 189,845   (afectadas: 175,676)


,exterior_color,interior_color
Beige,944,14276
Black,40524,106717
Blue,16412,496
Brown,1648,4300
Gold,545,16
Gray,30587,32714
Green,1666,25
Orange,913,77
Other,8185,12268
Purple,70,0


### 3.7 Modelo base

`model` mezcla el **modelo** con la **versión** (`'GLC 300 Base 4MATIC'`, `'Fusion SE'`): hay casi 8,000 valores distintos. Se crea `base_model` con el modelo comercial (`'GLC'`, `'Fusion'`) para el filtro del tablero y como entrada del modelo. `model` se conserva con el texto completo (la versión).

Regla: la **primera palabra**, salvo los modelos de varias palabras de la lista `MODELOS_COMPUESTOS` (`'Grand Cherokee'`, `'Range Rover Sport'`, `'Model 3'`…) y los `AMG` de Mercedes-Benz (`'AMG GLE'`), que son autos distintos de su versión normal.

In [17]:
# ── 3.7 Modelo base ────────────────────────────────────────────────────────

MODELOS_COMPUESTOS = [                                            # se revisan de la más larga a la más corta
    'Range Rover Sport', 'Range Rover Evoque', 'Range Rover Velar', 'Range Rover',
    'Grand Cherokee', 'Grand Caravan', 'Grand Wagoneer', 'Santa Fe', 'Santa Cruz',
    'Model 3', 'Model S', 'Model X', 'Model Y', 'Town & Country', 'Town Car', 'Land Cruiser',
    'New Beetle', 'Crown Victoria', 'Discovery Sport', 'Transit Connect', 'Mustang Mach-E', 'Monte Carlo',
]
MODELOS_COMPUESTOS = sorted(MODELOS_COMPUESTOS, key=len, reverse=True)

def modelo_base(x):
    for m in MODELOS_COMPUESTOS:
        if x == m or x.startswith(m + ' '):
            return m
    palabras = x.split()
    if palabras[0] == 'AMG' and len(palabras) > 1:
        return f'AMG {palabras[1]}'
    return palabras[0]


df['base_model'] = df['model'].map(modelo_base)

registrar('3.7', f"Crear base_model: {df['model'].nunique():,} modelos con versión → {df['base_model'].nunique():,} modelos base",
          'El modelo comercial es el nivel útil para filtrar en el tablero y para que el modelo generalice',
          len(df), len(df), len(df))

df.groupby('base_model')['model'].agg(['count', 'nunique']).sort_values('count', ascending=False).head(10)

[3.7] Crear base_model: 7,289 modelos con versión → 605 modelos base
       filas: 189,845 → 189,845   (afectadas: 189,845)


,count,nunique
base_model,,
Wrangler,2980,79
Sierra,2630,57
Silverado,2454,87
Rogue,2139,13
1500,2055,44
RAV4,1919,32
Grand Cherokee,1862,44
Fusion,1718,24
Corolla,1666,39


### 3.8 Imputar categorías desconocidas

Los nulos de `transmission`, `drivetrain` y `fuel_type` (los originales y los no reconocidos en 3.5) se **deducen del mismo modelo**: casi todos los autos de un mismo `manufacturer` + `base_model` comparten transmisión, tracción y combustible. Se usa el valor **más frecuente** (moda) del grupo. Si el grupo no tiene datos, se usa la moda de la marca y, al final, la moda general.

La función `imputar_por_grupo()` se reutiliza para los números del motor y del rendimiento (3.9 y 3.10), con la **mediana** en lugar de la moda.

In [18]:
# ── 3.8 Imputación por grupo ───────────────────────────────────────────────

def moda(serie):
    m = serie.mode()
    return m.iat[0] if len(m) else np.nan


def imputar_por_grupo(df, col, metodo, niveles):
    '''Rellena los nulos de 'col' con la moda o la mediana de cada grupo, del más específico al más general.
    niveles: lista de listas de columnas, por ejemplo [['manufacturer', 'base_model'], ['manufacturer']].
    Al final, los que sigan nulos se rellenan con el valor de toda la columna.'''
    estadistico = moda if metodo == 'moda' else 'median'
    for grupo in niveles:
        faltan = df[col].isna()
        if not faltan.any():
            break
        valor_grupo = df.groupby(grupo)[col].transform(estadistico)
        df.loc[faltan, col] = valor_grupo[faltan]
    faltan = df[col].isna()
    if faltan.any():
        df.loc[faltan, col] = moda(df[col]) if metodo == 'moda' else df[col].median()
    return df


NIVELES_MODELO = [['manufacturer', 'base_model'], ['manufacturer']]

for col in reglas:
    nulos_col = df[col].isna().sum()
    df = imputar_por_grupo(df, col, 'moda', NIVELES_MODELO)
    registrar('3.8', f'Imputar {col} con la moda de marca + modelo base',
              'El modelo no acepta nulos; los autos del mismo modelo comparten esta característica',
              len(df), len(df), nulos_col)

[3.8] Imputar transmission con la moda de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 2,372)


[3.8] Imputar drivetrain con la moda de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 4,598)


[3.8] Imputar fuel_type con la moda de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 5,459)


### 3.9 Motor: cilindrada, cilindros y turbo

`engine` es texto libre (`'2.0L I4 16V GDI DOHC Turbo'`). Se extraen tres atributos numéricos que sí puede usar el modelo:

| Nueva columna | Regla | Ejemplo |
|---|---|---|
| `engine_liters` | número seguido de `L` | `2.0L` → 2.0 |
| `cylinders` | `V`, `I`, `H`, `W` o `Flat` seguido de un número, o `N Cylinder` | `I4` → 4 |
| `turbo` | el texto contiene `turbo` o `ecoboost` | → True |

**Consistencia:** un auto **eléctrico** no tiene motor de combustión: se le asigna `engine_liters = 0`, `cylinders = 0` y `turbo = False`. Los demás nulos se imputan con la mediana (o la moda, para `turbo`) de marca + modelo base. `engine` se elimina después: su información ya quedó en las tres columnas.

In [19]:
# ── 3.9 Atributos del motor ────────────────────────────────────────────────

motor = df['engine'].fillna('')

litros = motor.str.extract(r'(\d{1,2}(?:\.\d)?)\s?-?L(?![a-z])', flags=re.I, expand=False).astype(float)
cil_1  = motor.str.extract(r'\b(?:V|I|H|W|Flat)-?\s?(\d{1,2})\b', flags=re.I, expand=False)
cil_2  = motor.str.extract(r'\b(\d{1,2})[- ]?Cyl', flags=re.I, expand=False)
cilindros = cil_1.fillna(cil_2).astype(float)

df['engine_liters'] = litros.where(litros.between(0.6, 8.5))          # fuera de rango → no interpretable
df['cylinders']     = cilindros.where(cilindros.between(2, 16))
df['turbo']         = motor.str.contains('turbo|ecoboost', case=False).where(df['engine'].notna())

interpretables = df[['engine_liters', 'cylinders']].notna().all(axis=1).sum()
registrar('3.9a', 'Extraer engine_liters, cylinders y turbo del texto de engine',
          'El modelo necesita números, no texto libre; cilindrada y cilindros explican buena parte del precio',
          len(df), len(df), interpretables)

# Consistencia: los eléctricos no tienen motor de combustión
es_electrico = df['fuel_type'] == 'Electric'
inconsistentes = (es_electrico & (df['cylinders'].fillna(0) > 0)).sum()
df.loc[es_electrico, ['engine_liters', 'cylinders']] = 0
df.loc[es_electrico, 'turbo'] = False
registrar('3.9b', 'Eléctricos: engine_liters = 0, cylinders = 0, turbo = False',
          'Un eléctrico no tiene motor de combustión; corrige los textos que mencionaban cilindros',
          len(df), len(df), es_electrico.sum())

# Imputar lo que no se pudo interpretar
for col, metodo in [('engine_liters', 'mediana'), ('cylinders', 'mediana'), ('turbo', 'moda')]:
    nulos_col = df[col].isna().sum()
    df = imputar_por_grupo(df, col, metodo, NIVELES_MODELO)
    registrar('3.9c', f'Imputar {col} con la {metodo} de marca + modelo base',
              'El modelo no acepta nulos; los autos del mismo modelo comparten motor',
              len(df), len(df), nulos_col)

df = df.drop(columns='engine')

print()
print(f'Eléctricos que mencionaban cilindros (corregidos): {inconsistentes:,}')
print()
df[['engine_liters', 'cylinders', 'turbo']].describe(include='all')

[3.9a] Extraer engine_liters, cylinders y turbo del texto de engine
       filas: 189,845 → 189,845   (afectadas: 177,917)
[3.9b] Eléctricos: engine_liters = 0, cylinders = 0, turbo = False
       filas: 189,845 → 189,845   (afectadas: 4,041)
[3.9c] Imputar engine_liters con la mediana de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 6,277)
[3.9c] Imputar cylinders con la mediana de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 5,230)


[3.9c] Imputar turbo con la moda de marca + modelo base
       filas: 189,845 → 189,845   (afectadas: 3,323)

Eléctricos que mencionaban cilindros (corregidos): 42



,engine_liters,cylinders,turbo
count,189845.000000,189845.000000,189845
unique,NaN,NaN,2
top,NaN,NaN,False
freq,NaN,NaN,127627
mean,3.018647,5.158729,NaN
std,1.416066,1.644843,NaN
min,0.000000,0.000000,NaN
25%,2.000000,4.000000,NaN
50%,2.500000,4.000000,NaN
75%,3.600000,6.000000,NaN


### 3.10 Rendimiento: ciudad y carretera

`mpg` viene como texto `'26-36'` (millas por galón en ciudad y en carretera). Se separa en `mpg_city` y `mpg_highway`. Cuando trae un solo número (`'22'`), se usa para ambos. Valores de 0 o mayores a 150 se consideran no válidos. Los nulos se imputan con la mediana de marca + modelo base; si no hay, con la del mismo tipo de combustible (un eléctrico rinde muy distinto a uno de gasolina).

In [20]:
# ── 3.10 Rendimiento ───────────────────────────────────────────────────────

partes = df['mpg'].str.extract(r'^(\d+(?:\.\d+)?)(?:-(\d+(?:\.\d+)?))?$').astype(float)
partes[1] = partes[1].fillna(partes[0])                               # un solo número → ciudad = carretera

df['mpg_city']    = partes[0].where(partes[0].between(5, 150))
df['mpg_highway'] = partes[1].where(partes[1].between(5, 150))

registrar('3.10a', 'Separar mpg en mpg_city y mpg_highway',
          'Dos números que el modelo puede usar en lugar de un texto',
          len(df), len(df), df[['mpg_city', 'mpg_highway']].notna().all(axis=1).sum())

for col in ['mpg_city', 'mpg_highway']:
    nulos_col = df[col].isna().sum()
    df = imputar_por_grupo(df, col, 'mediana', [['manufacturer', 'base_model'], ['fuel_type']])
    registrar('3.10b', f'Imputar {col} con la mediana de marca + modelo base (o del combustible)',
              'El modelo no acepta nulos; el rendimiento depende del modelo y del tipo de combustible',
              len(df), len(df), nulos_col)

df = df.drop(columns='mpg')
df[['mpg_city', 'mpg_highway']].describe().round(1)

[3.10a] Separar mpg en mpg_city y mpg_highway
       filas: 189,845 → 189,845   (afectadas: 155,137)
[3.10b] Imputar mpg_city con la mediana de marca + modelo base (o del combustible)
       filas: 189,845 → 189,845   (afectadas: 34,072)
[3.10b] Imputar mpg_highway con la mediana de marca + modelo base (o del combustible)
       filas: 189,845 → 189,845   (afectadas: 34,663)


,mpg_city,mpg_highway
count,189845.0,189845.0
mean,21.9,28.4
std,7.0,6.5
min,8.0,6.0
25%,17.0,24.0
50%,20.0,28.0
75%,25.0,32.0
max,126.0,111.0


### 3.11 Rebaja y columnas informativas

| Columna | Decisión | Por qué |
|---|---|---|
| `price_drop` | nulo → **0** | El valor mínimo registrado es $100: un nulo significa que el anuncio **no tuvo rebaja** |
| `seller_name` | nulo → `'Unknown'` | Es texto informativo para el tablero; `'Unknown'` deja claro que no hay dato |
| `seller_rating`, `driver_rating` | **se dejan nulas** | Son informativas: imputarlas sería inventar una calificación. El tablero muestra solo las que existen |

In [21]:
# ── 3.11 Rebaja y columnas informativas ────────────────────────────────────

nulos_col = df['price_drop'].isna().sum()
df['price_drop'] = df['price_drop'].fillna(0)
registrar('3.11a', 'price_drop: nulo → 0',
          'Nulo significa que no hubo rebaja (el mínimo registrado es $100)',
          len(df), len(df), nulos_col)

nulos_col = df['seller_name'].isna().sum()
df['seller_name'] = df['seller_name'].fillna('Unknown')
registrar('3.11b', "seller_name: nulo → 'Unknown' (las calificaciones se dejan nulas)",
          'Columnas solo informativas para el tablero: no se inventan calificaciones',
          len(df), len(df), nulos_col)

[3.11a] price_drop: nulo → 0
       filas: 189,845 → 189,845   (afectadas: 84,833)
[3.11b] seller_name: nulo → 'Unknown' (las calificaciones se dejan nulas)
       filas: 189,845 → 189,845   (afectadas: 9)


### 3.12 Tipos de dato

SQLite guardó enteros como decimal (`92945.0`) y las fechas como texto. Ahora que no hay nulos, cada columna toma su **tipo correcto**, y Parquet lo guardará dentro del archivo.

In [22]:
# ── 3.12 Tipos de dato ─────────────────────────────────────────────────────

tipos_antes = df.dtypes

df[['year', 'mileage', 'price', 'price_drop', 'cylinders']] = df[['year', 'mileage', 'price', 'price_drop', 'cylinders']].astype('int64')
df[COLS_BOOLEANAS + ['turbo']] = df[COLS_BOOLEANAS + ['turbo']].astype(bool)
df['engine_liters']      = df['engine_liters'].round(1)
df['driver_reviews_num'] = df['driver_reviews_num'].astype('Int64')       # entero que admite nulos
df['_extracted_at']      = pd.to_datetime(df['_extracted_at']).astype('datetime64[us]')

registrar('3.12', 'Convertir tipos (enteros, booleanos y fecha de extracción)',
          'Tipos correctos: el modelo y las consultas SQL tratan cada columna como lo que es',
          len(df), len(df), 0)

pd.DataFrame({'antes': tipos_antes, 'después': df.dtypes}).dropna().query('antes != después')

[3.12] Convertir tipos (enteros, booleanos y fecha de extracción)
       filas: 189,845 → 189,845   (afectadas: 0)


,antes,después
mileage,float64,int64
accidents_or_damage,float64,bool
one_owner,float64,bool
personal_use_only,float64,bool
driver_reviews_num,float64,Int64
price_drop,float64,int64
price,float64,int64
_extracted_at,str,datetime64[us]
cylinders,float64,int64
turbo,object,bool


### 3.13 Enriquecer con el catálogo de marcas (*join*)

**Enriquecer** es agregar información que el dataset no trae. Un catálogo pequeño con las 30 marcas agrega el **país de origen** y si es una marca **de lujo** o **masiva**: dos atributos que ayudan al modelo a separar precios y sirven como filtro en el tablero.

Dos cuidados en todo *join*:

1. **Que no multiplique filas:** `validate='m:1'` marca error si el catálogo tuviera una marca repetida.
2. **Registros sin pareja:** con `indicator=True` vemos qué marcas no están en el catálogo.

In [23]:
# ── 3.13 Catálogo de marcas ────────────────────────────────────────────────

catalogo_marcas = pd.DataFrame([
    ('Acura', 'Japan', 'Luxury'),        ('Audi', 'Germany', 'Luxury'),       ('BMW', 'Germany', 'Luxury'),
    ('Buick', 'USA', 'Mainstream'),      ('Cadillac', 'USA', 'Luxury'),       ('Chevrolet', 'USA', 'Mainstream'),
    ('Chrysler', 'USA', 'Mainstream'),   ('Dodge', 'USA', 'Mainstream'),      ('Ford', 'USA', 'Mainstream'),
    ('GMC', 'USA', 'Mainstream'),        ('Honda', 'Japan', 'Mainstream'),    ('Hyundai', 'South Korea', 'Mainstream'),
    ('INFINITI', 'Japan', 'Luxury'),     ('Jaguar', 'UK', 'Luxury'),          ('Jeep', 'USA', 'Mainstream'),
    ('Kia', 'South Korea', 'Mainstream'),('Land Rover', 'UK', 'Luxury'),      ('Lexus', 'Japan', 'Luxury'),
    ('Lincoln', 'USA', 'Luxury'),        ('Mazda', 'Japan', 'Mainstream'),    ('Mercedes-Benz', 'Germany', 'Luxury'),
    ('Mitsubishi', 'Japan', 'Mainstream'),('Nissan', 'Japan', 'Mainstream'),  ('Porsche', 'Germany', 'Luxury'),
    ('RAM', 'USA', 'Mainstream'),        ('Subaru', 'Japan', 'Mainstream'),   ('Tesla', 'USA', 'Luxury'),
    ('Toyota', 'Japan', 'Mainstream'),   ('Volkswagen', 'Germany', 'Mainstream'), ('Volvo', 'Sweden', 'Luxury'),
], columns=['manufacturer', 'brand_country', 'brand_segment'])

antes = len(df)
df = df.merge(catalogo_marcas, on='manufacturer', how='left', validate='m:1', indicator=True)

sin_pareja = df[df['_merge'] == 'left_only']
df = df.drop(columns='_merge')

registrar('3.13', 'Join con catálogo de marcas (brand_country, brand_segment)',
          'País y segmento de la marca ayudan al modelo a separar precios y son filtros útiles en el tablero',
          antes, len(df), len(sin_pareja))

print()
print(f'Filas antes / después del join : {antes:,} / {len(df):,}')
print(f'Marcas sin pareja en el catálogo: {sin_pareja["manufacturer"].unique().tolist()}')
print()

df.groupby(['brand_country', 'brand_segment']).size().unstack(fill_value=0)

[3.13] Join con catálogo de marcas (brand_country, brand_segment)
       filas: 189,845 → 189,845   (afectadas: 0)

Filas antes / después del join : 189,845 / 189,845
Marcas sin pareja en el catálogo: []



brand_segment,Luxury,Mainstream
brand_country,,
Germany,26088,6116
Japan,10525,48386
South Korea,0,14580
Sweden,2434,0
UK,3818,0
USA,8449,69449


### 3.14 Columnas derivadas

| Columna | Cálculo | Para qué |
|---|---|---|
| `age` | `2023 − year` (mínimo 0) | La antigüedad explica la depreciación mejor que el año |
| `mileage_per_year` | `mileage / age` (age mínima 1) | Distingue un auto muy usado de uno poco usado de la misma edad |
| `mileage_km` | `mileage × 1.609344` | Para mostrar kilómetros a usuarios en México |
| `original_price` | `price + price_drop` | Precio publicado antes de la rebaja (tablero) |
| `price_range` | rangos de precio | Filtro del tablero |

> `price_drop`, `original_price` y `price_range` se calculan a partir del precio o se conocen después de publicar: **no pueden ser entradas del modelo** (sería *data leakage*, le estaríamos dando la respuesta).

In [24]:
# ── 3.14 Columnas derivadas ────────────────────────────────────────────────

df['age']              = (ANIO_REFERENCIA - df['year']).clip(lower=0)
df['mileage_per_year'] = (df['mileage'] / df['age'].clip(lower=1)).round().astype('int64')
df['mileage_km']       = (df['mileage'] * 1.609344).round().astype('int64')
df['original_price']   = df['price'] + df['price_drop']
df['price_range']      = pd.cut(df['price'],
                                bins=[PRECIO_MIN, 15_000, 30_000, 50_000, 100_000, PRECIO_MAX],
                                labels=['$1k-15k', '$15k-30k', '$30k-50k', '$50k-100k', '$100k-200k'],
                                include_lowest=True).astype(str)

registrar('3.14', 'Columnas derivadas: age, mileage_per_year, mileage_km, original_price, price_range',
          'Antigüedad y uso anual explican la depreciación; km y rangos de precio sirven al tablero',
          len(df), len(df), 0)

df[['year', 'age', 'mileage', 'mileage_per_year', 'mileage_km', 'price', 'price_drop', 'original_price', 'price_range']].head()

[3.14] Columnas derivadas: age, mileage_per_year, mileage_km, original_price, price_range
       filas: 189,845 → 189,845   (afectadas: 0)


,year,age,mileage,mileage_per_year,mileage_km,price,price_drop,original_price,price_range
0,2013,10,92945,9294,149580,13988,300,14288,$1k-15k
1,2013,10,61539,6154,99037,16950,0,16950,$15k-30k
2,2017,6,15904,2651,25595,139876,2111,141987,$100k-200k
3,2017,6,20946,3491,33709,149995,0,149995,$100k-200k
4,2017,6,21074,3512,33915,127991,0,127991,$100k-200k


### 3.15 Metadatos de transformación y orden final

Igual que en EXTRACTED, agregamos cuándo y con qué versión de reglas se transformó cada fila. Los metadatos de extracción **se conservan**: con `_row_hash` cualquier fila limpia se rastrea hasta `extracted_autos` y `raw_autos`.

También se definen los **roles** de las columnas, que quedan guardados en el lineage para las siguientes capas.

In [25]:
# ── 3.15 Metadatos, roles y orden de columnas ─────────────────────────────

df['_transformed_at']    = pd.Timestamp(datetime.now().replace(microsecond=0)).as_unit('us')
df['_transform_version'] = VERSION_TRANSFORM

ROLES = {
    'objetivo'            : ['price'],
    'caracteristicas_auto': ['manufacturer', 'base_model', 'model', 'year', 'age', 'mileage', 'mileage_km',
                             'mileage_per_year', 'engine_liters', 'cylinders', 'turbo', 'transmission',
                             'drivetrain', 'fuel_type', 'mpg_city', 'mpg_highway', 'exterior_color',
                             'interior_color', 'accidents_or_damage', 'one_owner', 'personal_use_only',
                             'brand_country', 'brand_segment'],
    'solo_tablero_precio' : ['price_drop', 'original_price', 'price_range'],     # no son entradas del modelo
    'solo_tablero_info'   : COLS_INFORMATIVAS,                                  # no son entradas del modelo
    'metadatos'           : COLS_META + ['_transformed_at', '_transform_version'],
}

COLUMNAS_FINALES = (ROLES['caracteristicas_auto'] + ROLES['objetivo'] + ROLES['solo_tablero_precio']
                    + ROLES['solo_tablero_info'] + ROLES['metadatos'])

df_final = df[COLUMNAS_FINALES].reset_index(drop=True)

registrar('3.15', 'Metadatos de transformación y orden de columnas por rol',
          'Lineage: cada fila conserva su origen y su versión de reglas',
          len(df_final), len(df_final), 0)

[3.15] Metadatos de transformación y orden de columnas por rol
       filas: 189,845 → 189,845   (afectadas: 0)


In [26]:
inspeccionar_dataset(df_final, nombre=f'{NOMBRE_DESTINO} (antes de cargar)')


  DATASET: transformed_autos (antes de cargar)
  Filas    : 189,845
  Columnas : 37

  Tipos de datos y valores nulos:
    manufacturer                 str          nulos: 0 (0.0%)
    base_model                   str          nulos: 0 (0.0%)
    model                        str          nulos: 0 (0.0%)
    year                         int64        nulos: 0 (0.0%)
    age                          int64        nulos: 0 (0.0%)
    mileage                      int64        nulos: 0 (0.0%)
    mileage_km                   int64        nulos: 0 (0.0%)
    mileage_per_year             int64        nulos: 0 (0.0%)
    engine_liters                float64      nulos: 0 (0.0%)
    cylinders                    int64        nulos: 0 (0.0%)
    turbo                        bool         nulos: 0 (0.0%)
    transmission                 str          nulos: 0 (0.0%)
    drivetrain                   str          nulos: 0 (0.0%)
    fuel_type                    str          nulos: 0 (0.0%)
    mpg_city

,manufacturer,base_model,model,year,age,mileage,mileage_km,mileage_per_year,engine_liters,cylinders,turbo,transmission,drivetrain,fuel_type,mpg_city,mpg_highway,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,brand_country,brand_segment,price,price_drop,original_price,price_range,seller_name,seller_rating,driver_rating,driver_reviews_num,_batch_id,_extracted_at,_source_table,_row_hash,_transformed_at,_transform_version
0,Acura,ILX,ILX Hybrid 1.5L,2013,10,92945,149580,9294,1.5,4,False,Automatic,FWD,Gasoline,39.0,38.0,Black,Beige,False,False,False,Japan,Luxury,13988,300,14288,$1k-15k,Iconic Coach,NaN,4.4,12,1,2026-11-24 11:17:50,raw_autos,a5f41d2a94483372,2026-11-24 11:53:00,v1
1,Acura,ILX,ILX Hybrid 1.5L,2013,10,61539,99037,6154,1.5,4,False,CVT,FWD,Hybrid,39.0,38.0,Silver,Unknown,True,True,True,Japan,Luxury,16950,0,16950,$15k-30k,Napleton's Palatine Subaru,4.6,4.4,12,1,2026-11-24 11:17:50,raw_autos,d490101e1b4b1443,2026-11-24 11:53:00,v1
2,Acura,NSX,NSX Base,2017,6,15904,25595,2651,3.5,6,True,Automatic,AWD,Hybrid,21.0,22.0,White,Black,False,False,True,Japan,Luxury,139876,2111,141987,$100k-200k,Piazza Acura / Honda of Reading,4.7,4.8,17,1,2026-11-24 11:17:50,raw_autos,7497d846a2e6c6e0,2026-11-24 11:53:00,v1


In [27]:
# ── La bitácora completa ───────────────────────────────────────────────────

df_bitacora = pd.DataFrame(bitacora)

with pd.option_context('display.max_colwidth', 120):
    display(df_bitacora)

,paso,descripcion,justificacion,filas_antes,filas_despues,filas_afectadas
0,3.1,Quitar espacios extra en 10 columnas de texto,Valores iguales deben agruparse igual en el tablero y codificarse igual en el modelo,200000,200000,149
1,3.2,Eliminar duplicados exactos,Un anuncio repetido pesaría doble en el entrenamiento del modelo y en los conteos del tablero,200000,199388,612
2,3.3a,Eliminar filas sin millaje,El millaje es clave para el precio: no se puede inventar sin sesgar el modelo,199388,199263,125
3,3.3b,"Eliminar filas con nulos en ['accidents_or_damage', 'one_owner', 'personal_use_only']",Son entradas del formulario: imputarlas metería ruido en variables que el usuario sí conoce,199263,191073,8190
4,3.4a,"Eliminar precio fuera de [$1,000, $200,000]","Precios simbólicos, errores y superdeportivos distorsionan el modelo y no son autos que valúe el usuario",191073,190747,326
5,3.4b,"Eliminar año fuera de [2000, 2024]",Los clásicos se aprecian con la edad: siguen otra lógica de precio que confundiría al modelo,190747,189854,893
6,3.4c,"Eliminar millaje fuera de [0, 500,000]","Valores de relleno (999,999) que no son un millaje real",189854,189845,9
7,3.5,Estandarizar transmission: 539 variantes → 3 categorías (100 valores no reconocibles → nulo),Pocas categorías consistentes: el modelo las codifica bien y el tablero las usa como filtro,189845,189845,162557
8,3.5,Estandarizar drivetrain: 26 variantes → 4 categorías (16 valores no reconocibles → nulo),Pocas categorías consistentes: el modelo las codifica bien y el tablero las usa como filtro,189845,189845,182555
9,3.5,Estandarizar fuel_type: 20 variantes → 7 categorías (352 valores no reconocibles → nulo),Pocas categorías consistentes: el modelo las codifica bien y el tablero las usa como filtro,189845,189845,5167


---
## PASO 4 — Diagnóstico de calidad (después)

Repetimos **las mismas mediciones** del PASO 2 sobre `df_final`. Las columnas del auto ahora son las limpias (por ejemplo, `engine_liters` en lugar de `engine`).

In [28]:
# ── Mismas mediciones, ahora sobre df_final ────────────────────────────────

COLS_AUTO_FINAL = ROLES['caracteristicas_auto'] + ROLES['objetivo'] + ['price_drop']
COLS_NEGOCIO_FINAL = [c for c in df_final.columns if not c.startswith('_')]

variantes_despues = df_final[COLS_CATEGORICAS].nunique()
electricos_d = ((df_final['fuel_type'] == 'Electric') & (df_final['cylinders'] > 0)).sum()

calidad_despues = pd.Series({
    'filas'                          : len(df_final),
    'celdas_nulas_columnas_del_auto' : df_final[COLS_AUTO_FINAL].isna().sum().sum(),
    'celdas_nulas_informativas'      : df_final[COLS_INFORMATIVAS].isna().sum().sum(),
    'duplicados_exactos'             : df_final.duplicated(subset=COLS_NEGOCIO_FINAL).sum(),
    'precio_fuera_de_rango'          : (~df_final['price'].between(PRECIO_MIN, PRECIO_MAX)).sum(),
    'anio_fuera_de_rango'            : (~df_final['year'].between(ANIO_MIN, ANIO_MAX)).sum(),
    'millaje_nulo_o_fuera_de_rango'  : (~df_final['mileage'].between(MILLAJE_MIN, MILLAJE_MAX)).sum(),
    **{f'variantes_{c}': variantes_despues[c] for c in COLS_CATEGORICAS},
    'electricos_con_cilindros'       : electricos_d,
}, name='después')

df_calidad = pd.DataFrame({'antes': calidad_antes, 'después': calidad_despues}).astype('int64')
df_calidad['cambio'] = df_calidad['después'] - df_calidad['antes']
df_calidad

,antes,después,cambio
filas,200000,189845,-10155
celdas_nulas_columnas_del_auto,186091,0,-186091
celdas_nulas_informativas,66819,57441,-9378
duplicados_exactos,612,1,-611
precio_fuera_de_rango,377,0,-377
anio_fuera_de_rango,1536,0,-1536
millaje_nulo_o_fuera_de_rango,135,0,-135
variantes_transmission,696,3,-693
variantes_drivetrain,28,4,-24
variantes_fuel_type,23,7,-16


**Cómo leerla:**

- Nulos en las columnas del auto, precio, año y millaje fuera de rango, y eléctricos con cilindros deben quedar en **0**.
- `celdas_nulas_informativas` baja, pero **no llega a 0**: las calificaciones se dejan nulas a propósito (no se inventan). Las filas eliminadas en 3.2–3.4 también se llevan sus nulos.
- `duplicados_exactos` puede no quedar en 0: al estandarizar categorías y quitar columnas de texto, dos anuncios distintos (por ejemplo, con otro vendedor) pueden quedar iguales. Ya no son el mismo anuncio repetido, así que se conservan.
- Las variantes de cada categoría bajan de cientos o miles a unas pocas.
- La baja de `filas` debe coincidir con la suma de filas eliminadas en la bitácora.

In [29]:
# ── ¿Cuadra la bitácora con el conteo de filas? ────────────────────────────

eliminadas_bitacora = (df_bitacora['filas_antes'] - df_bitacora['filas_despues']).sum()
eliminadas_real     = len(df_ext) - len(df_final)

print()
print(f'Filas eliminadas según la bitácora : {eliminadas_bitacora:,}')
print(f'Filas eliminadas en realidad       : {eliminadas_real:,}')
print('Cuadra' if eliminadas_bitacora == eliminadas_real else 'NO CUADRA: revisa la bitácora')
print(f'Filas conservadas                  : {len(df_final):,} ({len(df_final) / len(df_ext) * 100:.1f} %)')
print()


Filas eliminadas según la bitácora : 10,155
Filas eliminadas en realidad       : 10,155
Cuadra
Filas conservadas                  : 189,845 (94.9 %)



---
## PASO 5 — Carga: escribir la capa TRANSFORMED en Parquet

Hasta aquí, `df_final` **solo vive en memoria**. La **carga** (la "L" de ETL) lo guarda en un destino persistente para que la siguiente capa lo consuma.

### ¿Por qué Parquet y no el `.db`?

- En la industria, la capa limpia (*silver*) vive normalmente en archivos **Parquet** dentro de un *data lake*. La base de datos para análisis llega después, al modelar.
- Parquet **guarda el tipo de cada columna** dentro del archivo: lo que hicimos en 3.12 no se pierde. SQLite y CSV no lo conservan.

### ¿Append o sobrescribir?

| Estrategia | Qué hace | ¿Dónde la usamos? |
|---|---|---|
| **Append** | Agrega filas nuevas al final | EXTRACTED: cada lote es historia que no se puede perder |
| **Sobrescribir** (*full refresh*) | Reemplaza el archivo completo | **TRANSFORMED**: se puede recalcular entero desde EXTRACTED |

Aquí **sobrescribimos**. Si cambia una regla de limpieza, basta con volver a ejecutar el notebook. A esto se le llama **idempotencia**: ejecutar el pipeline una o diez veces deja el mismo resultado.

In [30]:
# ── Log de operaciones (una línea JSON por operación) ──────────────────────

def escribir_log(etapa, filas_entrada, filas_salida, estado='exitoso', detalle=None):
    '''Agrega UNA línea al log de transformación.'''
    evento = {
        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'etapa'        : etapa,
        'origen'       : TABLA_EXTRACTED,
        'destino'      : ruta_relativa(PARQUET_DESTINO),
        'filas_entrada': filas_entrada,
        'filas_salida' : filas_salida,
        'version'      : VERSION_TRANSFORM,
        'estado'       : estado,
        'detalle'      : detalle,
    }
    with open(LOG_PATH, 'a', encoding='utf-8') as f:            # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')


print()
print('escribir_log() lista')
print()


escribir_log() lista



In [31]:
# ── Cargar df_final a Parquet ──────────────────────────────────────────────

try:
    df_final.to_parquet(PARQUET_DESTINO, index=False)           # si el archivo existe, se sobrescribe
    escribir_log('carga_parquet', len(df_ext), len(df_final), detalle=f'{tamano_mb(PARQUET_DESTINO):.2f} MB')
except Exception as e:
    escribir_log('carga_parquet', len(df_ext), 0, estado='fallido', detalle=str(e))
    raise

print()
print(f'{TABLA_EXTRACTED} → {NOMBRE_DESTINO}.parquet')
print(f'  Filas de entrada : {len(df_ext):,}')
print(f'  Filas cargadas   : {len(df_final):,}')
print(f'  Columnas         : {df_final.shape[1]}')
print(f'  Tamaño           : {tamano_mb(PARQUET_DESTINO):.2f} MB')
print()


extracted_autos → transformed_autos.parquet
  Filas de entrada : 200,000
  Filas cargadas   : 189,845
  Columnas         : 37
  Tamaño           : 8.69 MB



### ¿Se conservaron los tipos?

Leemos el archivo de vuelta y comparamos sus tipos con los de `df_final`. Deben ser **iguales**: esa es la ventaja de Parquet.

In [32]:
# ── Leer el Parquet de vuelta ──────────────────────────────────────────────

df_leido = pd.read_parquet(PARQUET_DESTINO)

tipos = pd.DataFrame({'en memoria': df_final.dtypes.astype(str), 'desde Parquet': df_leido.dtypes.astype(str)})
tipos['iguales'] = tipos['en memoria'] == tipos['desde Parquet']

print()
print(f'Filas leídas: {len(df_leido):,}  |  ¿Todos los tipos iguales? {tipos["iguales"].all()}')
print()

tipos


Filas leídas: 189,845  |  ¿Todos los tipos iguales? True



,en memoria,desde Parquet,iguales
manufacturer,str,str,True
base_model,str,str,True
model,str,str,True
year,int64,int64,True
age,int64,int64,True
mileage,int64,int64,True
mileage_km,int64,int64,True
mileage_per_year,int64,int64,True
engine_liters,float64,float64,True
cylinders,int64,int64,True


---
## PASO 6 — Lineage y evidencias

El **lineage** documenta el recorrido completo del dato: **origen → transformaciones → destino**. Se guarda como JSON junto al log, con la bitácora y los **roles** de las columnas, que la capa SERVING y el modelo usarán para saber qué columnas son entradas y cuáles solo informativas.

La bitácora y la comparación antes vs. después se guardan en `Reportes/` como CSV.

In [33]:
# ── Guardar el lineage ─────────────────────────────────────────────────────

lineage = {
    'recorrido'        : ['Kaggle (CSV)', 'raw_autos (SQLite)', f'{TABLA_EXTRACTED} (SQLite)', f'{NOMBRE_DESTINO} (Parquet)'],
    'origen'           : TABLA_EXTRACTED,
    'base_de_datos'    : ruta_relativa(DB_PATH),
    'filas_origen'     : len(df_ext),
    'destino'          : ruta_relativa(PARQUET_DESTINO),
    'filas_destino'    : len(df_final),
    'estrategia_carga' : 'sobrescribir (full refresh)',
    'version_reglas'   : VERSION_TRANSFORM,
    'generado_en'      : datetime.now().isoformat(timespec='seconds'),
    'trazabilidad'     : f'_row_hash vincula cada fila con {TABLA_EXTRACTED} y raw_autos',
    'reglas'           : {'precio': [PRECIO_MIN, PRECIO_MAX], 'anio': [ANIO_MIN, ANIO_MAX],
                          'millaje': [MILLAJE_MIN, MILLAJE_MAX], 'anio_referencia': ANIO_REFERENCIA},
    'roles_columnas'   : ROLES,
    'transformaciones' : bitacora,
    'columnas_destino' : df_final.dtypes.astype(str).to_dict(),
}

with open(LINEAGE_PATH, 'w', encoding='utf-8') as f:
    json.dump(lineage, f, indent=2, ensure_ascii=False, default=int)

print()
print(f'Lineage guardado en: {ruta_relativa(LINEAGE_PATH)}')
print()
print(' → '.join(lineage['recorrido']))
print()


Lineage guardado en: Transformacion_de_Datos/Logs/lineage_transformed_autos.json

Kaggle (CSV) → raw_autos (SQLite) → extracted_autos (SQLite) → transformed_autos (Parquet)



> `default=int` convierte a entero los números de pandas/NumPy (por ejemplo, un `int64`), que el módulo `json` no sabe escribir por sí solo.

In [34]:
# ── Guardar la bitácora y la calidad antes vs. después ─────────────────────

df_bitacora.to_csv(BITACORA_PATH, index=False, encoding='utf-8-sig')        # utf-8-sig: abre bien en Excel
df_calidad.to_csv(CALIDAD_PATH, index_label='metrica', encoding='utf-8-sig')

print()
print('Archivos guardados:')
print(f'  {ruta_relativa(BITACORA_PATH)}')
print(f'  {ruta_relativa(CALIDAD_PATH)}')
print(f'  {ruta_relativa(LOG_PATH)}')
print()


Archivos guardados:
  Transformacion_de_Datos/Reportes/bitacora_transformed_autos.csv
  Transformacion_de_Datos/Reportes/calidad_transformed_autos.csv
  Transformacion_de_Datos/Logs/transformacion_log.jsonl



---
## PASO 7 — Validación con DuckDB (sobre el Parquet ya cargado)

Una carga no está terminada hasta que **se valida lo que quedó en el destino**, no lo que había en memoria. Con **DuckDB** consultamos el archivo Parquet directamente con SQL, sin cargarlo a ninguna base.

In [35]:
# ── DuckDB ─────────────────────────────────────────────────────────────────

import duckdb

con = duckdb.connect()                                    # base en memoria: no crea ningún archivo
P   = f"read_parquet('{PARQUET_DESTINO}')"                # así se lee un Parquet dentro de SQL

print()
print(f'DuckDB {duckdb.__version__} listo')
print()


DuckDB 2.0.1 listo



In [36]:
# ── Conteo, rangos y nulos en el archivo cargado ───────────────────────────

con.sql(f'''
    SELECT COUNT(*)                                               AS filas,
           MIN(price)  AS precio_min,  MAX(price)  AS precio_max,
           MIN(year)   AS anio_min,    MAX(year)   AS anio_max,
           MAX(mileage)                                           AS millaje_max,
           SUM(CASE WHEN base_model    IS NULL THEN 1 ELSE 0 END) AS nulos_base_model,
           SUM(CASE WHEN engine_liters IS NULL THEN 1 ELSE 0 END) AS nulos_engine_liters,
           SUM(CASE WHEN transmission  IS NULL THEN 1 ELSE 0 END) AS nulos_transmission
    FROM {P}
''').df()

,filas,precio_min,precio_max,anio_min,anio_max,millaje_max,nulos_base_model,nulos_engine_liters,nulos_transmission
0,189845,1100,199999,2000,2023,481071,0.0,0.0,0.0


In [37]:
# ── Esquema: columnas y tipos guardados en el archivo ──────────────────────

con.sql(f'DESCRIBE SELECT * FROM {P}').df()[['column_name', 'column_type']]

,column_name,column_type
0,manufacturer,VARCHAR
1,base_model,VARCHAR
2,model,VARCHAR
3,year,BIGINT
4,age,BIGINT
5,mileage,BIGINT
6,mileage_km,BIGINT
7,mileage_per_year,BIGINT
8,engine_liters,DOUBLE
9,cylinders,BIGINT


In [38]:
# ── Primer consumo: precio mediano por marca y antigüedad ──────────────────

con.sql(f'''
    SELECT manufacturer,
           COUNT(*)                                                    AS anuncios,
           ROUND(MEDIAN(price))                                        AS precio_mediano,
           ROUND(MEDIAN(CASE WHEN age <= 3 THEN price END))            AS mediano_0_3_anios,
           ROUND(MEDIAN(CASE WHEN age BETWEEN 4 AND 8 THEN price END)) AS mediano_4_8_anios,
           ROUND(MEDIAN(CASE WHEN age >= 9 THEN price END))            AS mediano_9_o_mas
    FROM {P}
    GROUP BY manufacturer
    ORDER BY anuncios DESC
    LIMIT 10
''').df()

,manufacturer,anuncios,precio_mediano,mediano_0_3_anios,mediano_4_8_anios,mediano_9_o_mas
0,Ford,19755,29595.0,39997.0,23998.0,12991.0
1,Toyota,15085,28985.0,36991.0,26600.0,13538.0
2,Chevrolet,13855,26900.0,35091.0,23900.0,13825.0
3,Nissan,12287,22189.0,25997.0,19599.0,10995.0
4,Jeep,10461,27990.0,39998.0,22995.0,15369.0
5,Mercedes-Benz,9859,39896.0,48796.0,33195.0,16396.0
6,Honda,9469,24997.0,30989.0,23450.0,12491.0
7,BMW,9262,33795.0,47969.0,28998.0,14735.0
8,Kia,8995,20998.0,24460.0,17482.0,10491.0
9,GMC,7472,35322.0,49750.0,29995.0,14500.0


In [39]:
# ── Cerrar la conexión y liberar memoria ───────────────────────────────────

con.close()
del df_ext, df, df_leido

print()
print('Conexión cerrada y memoria liberada')
print()


Conexión cerrada y memoria liberada



---
### Resumen

| Paso | Qué hicimos |
|---|---|
| 1 | Leímos `extracted_autos` en **solo lectura** |
| 2 | **Diagnóstico** de completitud, unicidad, validez, estandarización y consistencia |
| 3 | **15 transformaciones** registradas en la bitácora con su justificación |
| 4 | Repetimos el diagnóstico y comparamos **antes vs. después** |
| 5 | **Carga** a `transformed_autos.parquet` (sobrescribir, idempotente) y verificación de tipos |
| 6 | **Lineage** (con los roles de las columnas), bitácora y calidad en CSV |
| 7 | **Validación** con DuckDB sobre el archivo cargado |

**Siguiente notebook:** `MODELADO.ipynb`, que leerá **`transformed_autos.parquet`** para construir el esquema estrella en `serving.db`.